# Introduction 

<font size="3"> Once the two-stages training of an I2IwFiLM model is terminated run this code in order to generate translated images of the test set. These results can be later processed for benchmarking purposes.</font>

# Code

<font size="3">Import required libraries and dependencies for the pipeline.</font>

In [ ]:
import os

from pathlib import Path
from omegaconf import OmegaConf
from hydra.utils import instantiate
import sys
sys.path.append('..')

from i2iwfilm.load import load_from_dir_without_hyperparam_in_statedict_new_pl

from datetime import datetime
import json

import os
import glob
import ipywidgets as widgets
import matplotlib
import matplotlib.pyplot as plt
import numpy as np
from skimage.metrics import structural_similarity as ssim

import sunpy.visualization.colormaps as cm

### Set environment variable and path to the image to be processed

<font size="3">Configure the I2IWFILM_PATH environment variable to the repository root.</font>

In [ ]:
# get the current directory
current_dir = Path.cwd()
# get the parent directory to get the root directory of the I2IwFiLM repository
parent_dir = current_dir.parent

# set env variable for the dataset path
os.environ["I2IWFILM_PATH"] = str(parent_dir)
print(os.environ["I2IWFILM_PATH"])

# Run model on test set

We define a main() function that adds image translation callback to list of testing callbacks on the an I2IwFiLM run and call its .test() function.

In [ ]:
def create_module_and_datamodule(cfg):
    datamodule = instantiate(cfg.dataset, _recursive_=False)
    module = instantiate(cfg.module, _recursive_=False) 
    return module, datamodule

def main(args):
    outpath = args.outpath
    model_path = args.model_path
    s1_run_dir = args.s1_run_dir
    s2_run_dir = args.s2_run_dir
    
    print(f"LOADING RUN: {args.run_dir}")
    st = datetime.now()
    run_dir = Path(args.run_dir)

    # set run_dirname as the name of the run directory (without the path)
    run_dirname = run_dir.name

    print(f"RUN DIRECTORY NAME: {run_dirname}")

    output_dir = run_dir / outpath

    print(f"OUTPUT DIRECTORY: {output_dir}")
    print(f"MODEL PATH: {model_path}")

    callbacks = {
        "test_reconstruction": {
            "_target_": "i2iwfilm.callback.I2IwFiLM_TestVizualizeImageTranslationCallback",
            "output_dir": str(output_dir),
            "batch_freq": 1,
            "max_images": "${dataset.batch_size}",
            "log_steps": [0],
            "clamp": True,
        }
    }
    callbacks = json.dumps(callbacks).replace(' ','').replace('"','')


    config, model, dm, trainer = load_from_dir_without_hyperparam_in_statedict_new_pl(
                                    run_path= run_dir,
                                    model_path= model_path,
                                    load_trainer=True,
                                    override = [
                                    f"++callbacks={callbacks}",
                                    #"trainer.enable_checkpointing=false",
                                    "logger=[]",
                                    f"model.s1_run_dir={s1_run_dir}",
                                    "++module.load_S1_weights=false",
                                    ]
                                )

    et = datetime.now()
    
    print(f"INITIALIZED RUN IN {et-st}")
    print("STARTING EVALUATION")
    st = datetime.now()
    results = trainer.test(model, datamodule=dm)
    et = datetime.now()
    print(f"EVALUATED RUN IN {et-st}")

Define a list of one or more paths to I2IwFiLM runs and call the main() function for each of the runs.

In [ ]:
runs_list = [
    # Define here relative path(s) to a run directory, each entry is a tuple (S2_run_dir, S1_run_dir).
    ("outputs/wl2cal/S2/I2IwFiLM_S2_whitelight2calcium", "outputs/wl2cal/S1/f1_I2IwFiLM_S1_seed_0"),
]

for i, p in enumerate(runs_list):
    run_path = os.environ["I2IWFILM_PATH"] + '/' + p[0]
    args = OmegaConf.create({
        "run_dir": run_path,
        "outpath": 'test_translation_viz',
        "model_path": 'last.ckpt',
        "s1_run_dir": os.environ["I2IWFILM_PATH"] + '/' + p[1],
        "s2_run_dir": run_path,
    })

    main(args)



# Visualize results

Now that the translated images of the test set are generated. We will now vizualize these images using an interactive interface with a graphical plot and an image selector. We first define the function that updates the graphical plot to show the source, target and generated images of the selected sample.

In [ ]:
%matplotlib ipympl

def show_result(value):
    global optionsSSIM

    sample = np.load(results_samples[res_slider.value])['arr_0']
    source = np.load(results_sources[res_slider.value])['arr_0']
    target = np.load(results_targets[res_slider.value])['arr_0']

    sample_name = os.path.basename(results_samples[res_slider.value])
    sample_name = sample_name.split('.')[0]
    source_name = os.path.basename(results_sources[res_slider.value])
    source_name = source_name.split('.')[0]
    target_name = os.path.basename(results_targets[res_slider.value])
    target_name = target_name.split('.')[0]

    
    diff = sample-target
    diff_abs = np.abs(diff)
    
    # ssim
    mean_ssim, ssim_img = ssim(sample, target, **optionsSSIM)
    
    ax[0,0].clear()
    ax[0,1].clear()
    ax[1,0].clear()
    ax[1,1].clear()
    
    cmap_304 = matplotlib.colormaps['sdoaia304']
    source = 1000*(source /255.)
    
    
    ax[0,0].imshow(sample, cmap = "gray", interpolation=None, vmin=0, vmax=255)
    ax[0,0].set_title(f'Output\n{sample_name}')
    ax[0,1].imshow(target, cmap = "gray", interpolation=None, vmin=0, vmax=255)
    ax[0,1].set_title(f'Target\n{target_name}')
    
    ax[1,0].imshow(source, cmap = cmap_304, interpolation=None)
    ax[1,0].set_title(f'Input\n{source_name}')  
    ax[1,1].imshow(ssim_img, cmap='Reds_r',)
    ax[1,1].set_title(f'SSIM Map (mean: {mean_ssim:.3f})')

In [ ]:
os.environ["I2IWFILM_PATH"] = str(parent_dir)

runs_list = [
    "outputs/wl2cal/S2/I2IwFiLM_S2_whitelight2calcium", 
]
p = runs_list[0]

results_path = os.environ["I2IWFILM_PATH"] + '/' + p + '/test_translation_viz/npz'

results_samples = sorted(glob.glob(results_path + '/*_sample.npz'))
results_sources = sorted(glob.glob(results_path + '/*_source.npz'))
results_targets = sorted(glob.glob(results_path + '/*_target.npz'))

optionsSSIM = {
                'full':True, 
                'win_size': 11, 
                'gaussian_weights': False,
                'sigma': 1.5,
                'k1': 0.01,
                'k2': 0.03,
                'data_range':255.
                }

res_slider = widgets.IntSlider(min=0, max=len(results_samples)-1, step=1, value=0)
res_slider.observe(show_result, names='value')

plt.ioff()
fig, ax = plt.subplots(2,2, figsize=(8,8))
plt.ion()

show_result(None)

widgets.VBox([res_slider, 
              fig.canvas,
             ])
